# RAG 튜토리얼 — 청킹 & 임베딩 비교

실험 변수는 딱 두 가지입니다.

| 변수 | 선택지 |
|---|---|
| **청킹 방식** | 고정 크기 / 문단 기준 / 슬라이딩 윈도우 |
| **임베딩 모델** | OpenAI / bge-m3 (로컬) |

이 노트북은 각 방식의 결과를 **나란히 비교**합니다.  
비교 결과를 보고 본인이 선택한 방식을 `{이름}_exp01.py`에 구현하세요.

---
**고정값** (모든 팀원 동일)
- 질문 셋: `COMMON_QUESTIONS` (5개)
- LLM: OpenAI gpt-4o-mini
- 답변 형식: 한국어, 근거 문서 기반

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("../.env"))
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
print("API 키:", "OK" if OPENAI_API_KEY else "없음 — .env 확인 필요")

In [ ]:
import fitz  # PyMuPDF

# 비교에 쓸 PDF — 본인 경로로 변경
PDF_PATH = "../data/lg/aircon/SQ09GK1WEN.pdf"

doc = fitz.open(PDF_PATH)
full_text = "".join(page.get_text() for page in doc)
print(f"전체 텍스트 길이: {len(full_text):,}자 / 총 {len(doc)}페이지")

---
## Part 1. 청킹 방식 비교

같은 PDF를 세 가지 방식으로 나눠서 결과를 비교합니다.

In [ ]:
# ── 방식 A: 고정 크기 (Fixed Size) ──────────────────────────────────────────
# 텍스트를 일정 길이로 기계적으로 자름. 구현이 가장 단순하지만 문맥이 잘릴 수 있음.

def chunk_fixed(text: str, size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks_A = chunk_fixed(full_text, size=500, overlap=50)
print(f"[A] 고정 크기 500자: {len(chunks_A)}개")
print(f"    최소={min(len(c) for c in chunks_A)}자 / 최대={max(len(c) for c in chunks_A)}자")
print(f"\n예시:")
print(chunks_A[3])

In [ ]:
# ── 방식 B: 문단 기준 (Paragraph) ───────────────────────────────────────────
# 빈 줄(\n\n)을 경계로 자름. 문단 단위라 문맥이 더 잘 보존됨.
# 단, 문단 길이가 들쭉날쭉할 수 있음.

def chunk_paragraph(text: str, max_size: int = 800) -> list[str]:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) <= max_size:
            current += ("\n\n" if current else "") + para
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks

chunks_B = chunk_paragraph(full_text, max_size=800)
print(f"[B] 문단 기준 (최대 800자): {len(chunks_B)}개")
print(f"    최소={min(len(c) for c in chunks_B)}자 / 최대={max(len(c) for c in chunks_B)}자")
print(f"\n예시:")
print(chunks_B[3])

In [ ]:
# ── 방식 C: 슬라이딩 윈도우 (Sliding Window) ────────────────────────────────
# 오버랩을 크게 줘서 문맥이 잘리지 않게 함.
# 청크 수가 많아지고 중복 내용이 늘어남.

def chunk_sliding(text: str, size: int = 400, overlap: int = 200) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks_C = chunk_sliding(full_text, size=400, overlap=200)
print(f"[C] 슬라이딩 윈도우 (400자, 오버랩 200): {len(chunks_C)}개")
print(f"    최소={min(len(c) for c in chunks_C)}자 / 최대={max(len(c) for c in chunks_C)}자")
print(f"\n예시:")
print(chunks_C[3])

In [ ]:
# 청킹 방식 요약 비교
print("=" * 50)
print(f"{'방식':<20} {'청크 수':>8} {'평균 길이':>10} {'최대 길이':>10}")
print("-" * 50)
for label, chunks in [("A. 고정 크기", chunks_A), ("B. 문단 기준", chunks_B), ("C. 슬라이딩", chunks_C)]:
    sizes = [len(c) for c in chunks]
    print(f"{label:<20} {len(chunks):>8} {sum(sizes)/len(sizes):>10.0f} {max(sizes):>10}")
print("=" * 50)
print()
print("→ 청크 수가 많을수록 검색 범위는 넓어지지만 노이즈도 늘어납니다.")
print("→ 평균 길이가 너무 짧으면 문맥이 부족하고, 너무 길면 검색 정확도가 떨어집니다.")

---
## Part 2. 임베딩 모델 비교

같은 텍스트를 두 모델로 임베딩해서 의미 유사도를 비교합니다.

In [ ]:
import numpy as np

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# 비교용 문장 쌍
TEST_PAIRS = [
    ("에어컨 필터 청소 방법",  "에어컨 필터 세척하는 법",   "유사한 의미"),
    ("에어컨 필터 청소 방법",  "filter cleaning air conditioner", "한영 같은 의미"),
    ("에어컨 필터 청소 방법",  "냉장고 온도 설정",            "다른 주제"),
    ("UE 오류",                "언밸런스 에러",               "다른 표현 같은 의미"),
]

In [ ]:
# ── 모델 1: OpenAI text-embedding-3-small ────────────────────────────────────
from openai import OpenAI

oai = OpenAI(api_key=OPENAI_API_KEY)

def embed_openai(texts: list[str]) -> list[list[float]]:
    resp = oai.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in resp.data]

print("[OpenAI] 유사도 비교")
print("-" * 55)
for a, b, label in TEST_PAIRS:
    vecs = embed_openai([a, b])
    sim = cosine_sim(vecs[0], vecs[1])
    print(f"{label:<18} {sim:.4f}   {a[:15]!r} ↔ {b[:15]!r}")

In [ ]:
# ── 모델 2: bge-m3 (로컬, 한국어 특화) ─────────────────────────────────────
# 최초 실행 시 4.3GB 모델 자동 다운로드 (약 10분 소요)

from sentence_transformers import SentenceTransformer

bge = SentenceTransformer("BAAI/bge-m3")

def embed_bge(texts: list[str]) -> list[list[float]]:
    return bge.encode(texts, normalize_embeddings=True).tolist()

print("[bge-m3] 유사도 비교")
print("-" * 55)
for a, b, label in TEST_PAIRS:
    vecs = embed_bge([a, b])
    sim = cosine_sim(vecs[0], vecs[1])
    print(f"{label:<18} {sim:.4f}   {a[:15]!r} ↔ {b[:15]!r}")

In [ ]:
# 두 모델 나란히 비교
print(f"{'케이스':<20} {'OpenAI':>8} {'bge-m3':>8}")
print("-" * 38)
for a, b, label in TEST_PAIRS:
    sim_oai = cosine_sim(embed_openai([a, b])[0], embed_openai([a, b])[1])
    sim_bge = cosine_sim(embed_bge([a, b])[0], embed_bge([a, b])[1])
    print(f"{label:<20} {sim_oai:>8.4f} {sim_bge:>8.4f}")
print()
print("→ 한국어 전문 용어(UE 오류↔언밸런스)에서 두 모델 차이가 드러납니다.")

---
## Part 3. 조합별 검색 결과 비교

청킹 방식 × 임베딩 모델 조합으로 같은 질문에 어떤 결과가 나오는지 비교합니다.

In [ ]:
import chromadb

def build_collection(collection_name: str, chunks: list[str], embed_fn) -> chromadb.Collection:
    """청크 + 임베딩 함수로 인메모리 ChromaDB 컬렉션 생성."""
    client_db = chromadb.Client()
    col = client_db.get_or_create_collection(collection_name)
    
    # 배치 처리 (임베딩 API 호출 최소화)
    batch_size = 20
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        vecs = embed_fn(batch)
        col.add(
            ids=[f"chunk_{i+j}" for j in range(len(batch))],
            embeddings=vecs,
            documents=batch,
        )
    return col

def search(col: chromadb.Collection, query: str, embed_fn, top_k: int = 3):
    [q_vec] = embed_fn([query])
    res = col.query(query_embeddings=[q_vec], n_results=top_k)
    return list(zip(res["documents"][0], res["distances"][0]))

print("컬렉션 빌드 함수 준비 완료")

In [ ]:
# 비교할 조합 (시간/비용 상 샘플만)
# 실제 실험에서는 전체 PDF 사용
SAMPLE_CHUNKS_A = chunks_A[:50]  # 고정 크기 50개만
SAMPLE_CHUNKS_B = chunks_B[:50]  # 문단 기준 50개만

print("인덱스 생성 중 (OpenAI API 호출)...")
col_A_oai = build_collection("A_openai", SAMPLE_CHUNKS_A, embed_openai)
col_B_oai = build_collection("B_openai", SAMPLE_CHUNKS_B, embed_openai)
print("완료")

In [ ]:
# 같은 질문으로 두 조합 검색 결과 비교
QUERY = "필터 청소 방법을 알려줘"

print(f"질문: {QUERY!r}")
print()

for label, col in [("A. 고정크기+OpenAI", col_A_oai), ("B. 문단기준+OpenAI", col_B_oai)]:
    results = search(col, QUERY, embed_openai)
    print(f"[{label}]")
    for i, (text, dist) in enumerate(results, 1):
        print(f"  {i}. distance={dist:.4f} | {text[:80].strip()}...")
    print()

---
## 다음 단계

비교 결과를 보고 본인이 선택한 방식을 `{이름}_exp01.py`에 구현하세요.

**구현할 함수는 하나입니다:**

```python
def my_answer(query: str) -> dict:
    # 청킹 → 임베딩 → DB 저장 → 검색 → LLM 호출 전부 여기서 자유롭게
    # LG·삼성 PDF 모두 data/ 아래에 있음 (lg/ + samsung/)
    ...
    return {"answer": str, "candidates": list}
```

나머지(평가 루프, JSON 저장)는 공통 harness가 자동으로 처리합니다.